In [1]:
!pip install scikit-learn-intelex

In [2]:
import os
import gzip
import pickle
import numpy as np
import pandas as pd

import seaborn as sns
import matplotlib
matplotlib.use('Agg')  # to save figures without a GUI
import matplotlib.pyplot as plt

from sklearn.svm import SVC
from sklearn.metrics import confusion_matrix, classification_report

In [3]:
print("Loading dataset...")
with gzip.open('dataset/train_set.pkl', 'rb') as f:
    train_set = pickle.load(f)
with gzip.open('dataset/test_set.pkl', 'rb') as f:
    test_set = pickle.load(f)
print("Dataset loaded!")

# Drop any NaN rows
train_set = train_set.dropna()
test_set  = test_set.dropna()

# We assume the columns include 'race_id', 'top20', and some numeric features.
features = ['performance_wma', 'top20_consistency', 'races_dby_consistency']
print("Features:", features)

# Build train_df and test_df with the columns we need: race_id, the features, and top20
train_df = train_set[['race_id'] + features + ['top20']].copy()
test_df  = test_set[['race_id'] + features + ['top20']].copy()

# Separate X and y for training
X_train = train_df[features]
y_train = train_df['top20']

# Separate X and y for testing
X_test = test_df[features]
y_test = test_df['top20']

Loading dataset...
Dataset loaded!
Features: ['performance_wma', 'top20_consistency', 'races_dby_consistency']


In [4]:
from sklearnex import patch_sklearn
patch_sklearn()

Intel(R) Extension for Scikit-learn* enabled (https://github.com/intel/scikit-learn-intelex)


In [ ]:
def main():
    print("Starting SVM training...")
    # Create and train the SVM
    # We set probability=True so we can use predict_proba (slower, but useful for ranking)
    svm_model = SVC(probability=True, random_state=42)
    svm_model.fit(X_train, y_train)

    print("SVM trained.")
    print("Performing inference on the test set...")

    # Compute probabilities for the positive class (top20)
    pred_probs_test = svm_model.predict_proba(X_test)[:, 1]
    # Add a 'pred_prob' column to test_df
    test_df['pred_prob'] = pred_probs_test

    # Function to select the top 20 rows for each race based on predicted probability
    def evaluate_top_20(df, race_id_col='race_id', prob_col='pred_prob'):
        grouped_results = []
        for rid, group in df.groupby(race_id_col):
            # Sort in descending order by probability
            group = group.sort_values(by=prob_col, ascending=False).copy()
            group['predicted_top20'] = 0
            top_n = min(20, len(group))
            group.iloc[:top_n, group.columns.get_loc('predicted_top20')] = 1
            grouped_results.append(group)
        return pd.concat(grouped_results)

    # Apply the top20 logic on the test set
    test_df = evaluate_top_20(test_df)

    # Extract y_true and y_pred
    y_true_test = test_df['top20'].values
    y_pred_test = test_df['predicted_top20'].values

    # Confusion matrix and classification report
    cm_test = confusion_matrix(y_true_test, y_pred_test)
    cr_test = classification_report(y_true_test, y_pred_test)

    print("Confusion Matrix - Test\n", cm_test)
    print("Classification Report - Test\n", cr_test)

    # Save figures and reports
    os.makedirs("models/SVM-no-sampling", exist_ok=True)

    plt.figure(figsize=(10,7))
    sns.heatmap(cm_test, annot=True, fmt='d', cmap='Blues')
    plt.title('Confusion Matrix - Test (SVM, no sampling)')
    plt.xlabel('Predicted')
    plt.ylabel('Actual')
    plt.savefig("models/SVM-no-sampling/ConfusionMatrix-Test.png")
    plt.clf()

    with open("models/SVM-no-sampling/scores-test.txt", "w") as f:
        f.write(cr_test)

    print("Done. Results saved in 'models/SVM-no-sampling/'.")

if __name__ == "__main__":
    main()

Starting SVM training...
